# Farm Crop-Mix Optimization — v2 (52-Week Weekly Model)

**What this notebook does:** Finds the profit-maximizing planting schedule
for a farm with user-defined crops, channels, and resource constraints.

**Key improvements over v1:**
- Weekly resolution (52 weeks) instead of 12 monthly buckets
- User-defined crops — any number, defined by days-to-maturity and season windows
- Succession calendar auto-generated (one start slot per week)
- Multiple sales channels with per-channel pricing and demand caps
- Detailed water model: rainfall, catchment tank, well, optional city water
- Detailed labor model: salaried vs hourly staff, seasonal crews, overtime

**Assumptions:** See `SPEC_v2.md` Section 2 for the full list of 16 assumptions.

---
**How to run:** Execute cells top-to-bottom.
Edit the parameter cells (Sections 3A–3I) to customise the model for your farm.

## Setup

In [ ]:
import sys, os
# Make sure the farm_v2 package is importable from the excel-app directory
if os.path.basename(os.getcwd()) != 'excel-app':
    os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from farm_v2.utils.water  import get_rainfall_weekly
from farm_v2.utils.calendar import generate_successions, week_start_date, week_end_date
from farm_v2.validate import validate_params
from farm_v2.solver   import solve_farm_v2

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Setup complete.')

---
## Section 3: Input Parameters

Edit the cells below to match your farm.  Each sub-section has comments
explaining every parameter.  You do not need to touch the solver code.

### 3A. Farm Infrastructure

In [ ]:
# ---- 3A. Farm Infrastructure ----
TOTAL_LAND          = 150       # acres  — total available farmland
HARVEST_STORAGE_CAP = 24_000    # m³     — weekly cold/staging storage capacity
EQUIP_STORAGE       = 500       # m³     — equipment shed / general storage
FUEL_STORAGE        = 25_000    # liters — on-farm fuel tank capacity
WATER_STORAGE_CAPACITY = 500_000  # liters — catchment tank (0 if no tank)

# --- Cash / Financing ---
# The model tracks weekly cumulative cash and enforces that the farm never
# draws more than CREDIT_LINE beyond its opening cash balance.
# Increase CREDIT_LINE to allow plans that require more pre-harvest financing.
OPENING_CASH = 150_000   # $ cash on hand at start of year (equity / savings)
CREDIT_LINE  = 600_000   # $ maximum operating line of credit from lender
ANNUAL_INTEREST_RATE = 0.07   # informational only — shown in cash flow chart

### 3B. Fixed Cost Unit Rates

In [ ]:
# ---- 3B. Fixed Cost Rates ----
# All rates are $/unit/week.  Annual cost = rate × capacity × OPERATING_WEEKS.
OPERATING_WEEKS          = 48      # weeks/year the farm incurs infrastructure costs

RATE_HARVEST_STORAGE     = 0.356   # $/m³/week    → ~$406k/yr at 24,000 m³ × 48 wks
RATE_WATER_STORAGE       = 0.00360 # $/liter/week → ~$86k/yr  at 500,000 L  × 48 wks
RATE_IRRIGATION          = 38.54   # $/acre/week  → ~$278k/yr at 150 acres   × 48 wks
RATE_EQUIP_STORAGE       = 1.042   # $/m³/week    → ~$25k/yr  at 500 m³      × 48 wks
RATE_FUEL_STORAGE        = 0.0208  # $/liter/week → ~$25k/yr  at 25,000 L    × 48 wks
RATE_MACHINERY           = 4_167   # $/week       → ~$200k/yr over 48 wks

### 3C. Water Configuration

In [ ]:
# ---- 3C. Water Configuration ----
# Choose your location for built-in historical weekly rainfall averages.
# Options: 'california_central_valley', 'pacific_northwest', 'southeast',
#          'midwest', 'northeast', 'southwest_desert'
# Or supply your own 52-element array in liters/acre/week.
CLIMATE_ZONE = 'california_central_valley'
RAINFALL_WEEKLY = get_rainfall_weekly(CLIMATE_ZONE)

# Catchment tank (collects roof/pond runoff)
CATCHMENT_ENABLED           = True
CATCHMENT_SURFACE_AREA_M2   = 3_000    # m² of impervious catchment surface
# CATCHMENT_EFFICIENCY hard-coded at 0.85 in water.py

# Well
WELL_ENABLED        = True
# 2.5M L/week ≈ realistic deep-well supply for 150-acre California vegetable farm
# (150 acres × ~17M L/acre/year = 49M L/year → ~940k L/week average demand)
WELL_WEEKLY_CAP_L   = 2_500_000  # liters/week year-round cap
WELL_COST_PER_L     = 0.00015    # $/liter pumping cost

# City water (purchased when free sources run short)
CITY_WATER_ENABLED          = False
CITY_WATER_COST_PER_L       = 0.0008   # $/liter (only charged when used)
CITY_WATER_FIXED_FEE_PER_WEEK = 0.0    # $/week standing connection fee

print(f'Peak weekly rainfall: {RAINFALL_WEEKLY.max():.0f} L/acre/week')
print(f'Well supply: {WELL_WEEKLY_CAP_L:,} L/week')

### 3D. Labor Configuration

In [ ]:
# ---- 3D. Labor Configuration ----
# type: 'salary' (fixed annual cost) or 'hourly' (variable cost, adds to labor capacity)
YEAR_ROUND_EMPLOYEES = [
    {
        'position':       'Farm Manager',
        'count':          1,
        'type':           'salary',
        'wage':           120_000,   # $/year
        'hours_per_week': 50,
        'benefits_pct':   0.25,      # 25% benefits overhead
    },
    {
        'position':       'Assistant Manager',
        'count':          2,
        'type':           'salary',
        'wage':           65_000,
        'hours_per_week': 45,
        'benefits_pct':   0.25,
    },
    {
        'position':       'Equipment Operator',
        'count':          3,
        'type':           'hourly',
        'wage':           24.00,     # $/hour
        'hours_per_week': 40,
        'benefits_pct':   0.20,
    },
]

# type: 'FT' (full-time) or 'PT' (part-time) — both are hourly, seasonal
SEASONAL_EMPLOYEES = [
    {
        'position':        'Field Crew (spring/summer)',
        'count':           25,
        'type':            'FT',
        'start_week':      10,   # mid-March
        'end_week':        40,   # early October
        'wage':            16.00,
        'hours_per_week':  45,
        'benefits_pct':    0.10,
        'piece_rate_per_lb': 0.0,  # set > 0 to add harvest piece-rate cost
    },
    {
        'position':        'Field Crew (fall harvest)',
        'count':           12,
        'type':            'FT',
        'start_week':      38,   # late September
        'end_week':        48,   # late November
        'wage':            16.00,
        'hours_per_week':  45,
        'benefits_pct':    0.10,
        'piece_rate_per_lb': 0.0,
    },
]

# Overtime
OVERTIME_ENABLED          = False
OVERTIME_THRESHOLD_HRS    = 40.0
OVERTIME_RATE_MULTIPLIER  = 1.5

### 3E. Equipment Constraints

In [ ]:
# ---- 3E. Equipment Proxy Constraints ----
# Maximum land that can physically be planted or harvested in any single week.
# Set to TOTAL_LAND to disable (no constraint beyond land itself).
MAX_PLANT_ACRES_PER_WEEK   = 60.0   # acres/week
MAX_HARVEST_ACRES_PER_WEEK = 80.0   # acres/week

### 3F. Sales Channels

In [ ]:
# ---- 3F. Sales Channels ----
# Define any number of channels.  Each crop in Section 3G must list its
# price, packaging cost, and demand cap for each channel you define here.
CHANNELS = [
    {'name': 'Wholesale'},
    {'name': 'Farmers Market'},
]

### 3G. Crop Definitions

This is the primary user-editable section.

**Cycle fields:**
- `days_to_maturity` — total crop cycle length; `total_weeks = ceil(days / 7)`
- `planting_weeks` / `harvest_weeks` — phases at start/end of cycle
- `growing_weeks` is derived automatically

**Season windows:** list of `{"start": "MM-DD", "end": "MM-DD"}` pairs.
One succession slot is auto-generated per start week that fits the cycle.

**Channel fields** (one entry per channel name):
- `price_per_lb` — selling price before packaging
- `packaging_cost_per_lb` — packaging + handling (can include piece-rate)
- `demand_cap_type` — `"annual"`, `"weekly_flat"`, or `"weekly_varying"`

In [ ]:
# ---- 3G. Crop Definitions ----
# Tip: add/remove crops here freely.  The succession calendar and solver
# adapt automatically.

CROPS = [
    # ------------------------------------------------------------------
    {
        'name': 'Spinach',
        'days_to_maturity': 42,      # 6 weeks: 1 plant + 3 grow + 2 harvest
        'planting_weeks':   1,
        'harvest_weeks':    2,
        'seasons': [
            {'start': '02-01', 'end': '06-15'},   # spring
            {'start': '08-01', 'end': '11-15'},   # fall
        ],
        'min_acreage_if_planted':  2.0,   # acres
        'crop_diversity_max':      0.50,  # fraction of total land (≤50% to spinach at once)
        'yield_per_acre':          3_250, # lbs/acre/harvest-week
        'crop_size_m3_per_lb':     0.0022,
        'planting_type':           'seed',
        'seed_transplant_cost_per_acre': 280,
        # Resource use per acre per week, by phase (plant / grow / harvest)
        'labor_plant': 0.7,  'labor_grow': 0.5,  'labor_harv': 1.9,  # hrs/acre/wk
        'water_plant': 10_400, 'water_grow': 12_700, 'water_harv': 9_200,  # L/acre/wk
        'fert_plant':  0.46,  'fert_grow':  0.35,  'fert_harv':  0.23,  # lbs/acre/wk
        'fuel_plant':  3.5,   'fuel_grow':  1.8,   'fuel_harv':  4.6,   # L/acre/wk
        'channels': {
            'Wholesale': {
                'price_per_lb':           1.10,
                'packaging_cost_per_lb':  0.09,
                'demand_cap_type':        'annual',
                'annual_demand_lbs':      1_200_000,
            },
            'Farmers Market': {
                'price_per_lb':           2.10,
                'packaging_cost_per_lb':  0.18,
                'demand_cap_type':        'weekly_flat',
                'weekly_demand_lbs':      6_000,   # ~312k lbs/year
            },
        },
    },
    # ------------------------------------------------------------------
    {
        'name': 'Cucumbers',
        'days_to_maturity': 70,      # 10 weeks: 1 plant + 6 grow + 3 harvest
        'planting_weeks':   1,
        'harvest_weeks':    3,
        'seasons': [
            {'start': '05-01', 'end': '09-01'},
        ],
        'min_acreage_if_planted':  2.0,
        'crop_diversity_max':      0.40,
        'yield_per_acre':          3_000,
        'crop_size_m3_per_lb':     0.0030,
        'planting_type':           'seed',
        'seed_transplant_cost_per_acre': 190,
        'labor_plant': 0.9,  'labor_grow': 0.7,  'labor_harv': 2.3,
        'water_plant': 13_900, 'water_grow': 19_600, 'water_harv': 17_300,
        'fert_plant':  0.69,  'fert_grow':  0.46,  'fert_harv':  0.23,
        'fuel_plant':  4.2,   'fuel_grow':  2.3,   'fuel_harv':  5.1,
        'channels': {
            'Wholesale': {
                'price_per_lb':           0.55,
                'packaging_cost_per_lb':  0.06,
                'demand_cap_type':        'annual',
                'annual_demand_lbs':      900_000,
            },
            'Farmers Market': {
                'price_per_lb':           1.05,
                'packaging_cost_per_lb':  0.12,
                'demand_cap_type':        'weekly_varying',
                'demand_ranges': [
                    {'weeks': 'W18-W40', 'weekly_lbs': 4_000},
                ],
            },
        },
    },
    # ------------------------------------------------------------------
    {
        'name': 'Sweet Corn',
        'days_to_maturity': 84,      # 12 weeks: 1 plant + 9 grow + 2 harvest
        'planting_weeks':   1,
        'harvest_weeks':    2,
        'seasons': [
            {'start': '04-15', 'end': '09-01'},
        ],
        'min_acreage_if_planted':  3.0,
        'crop_diversity_max':      0.35,
        'yield_per_acre':          6_000,
        'crop_size_m3_per_lb':     0.0015,
        'planting_type':           'seed',
        'seed_transplant_cost_per_acre': 120,
        'labor_plant': 0.9,  'labor_grow': 0.5,  'labor_harv': 2.3,
        'water_plant': 11_500, 'water_grow': 18_500, 'water_harv': 15_000,
        'fert_plant':  0.58,  'fert_grow':  0.46,  'fert_harv':  0.35,
        'fuel_plant':  4.6,   'fuel_grow':  2.3,   'fuel_harv':  5.8,
        'channels': {
            'Wholesale': {
                'price_per_lb':           0.38,
                'packaging_cost_per_lb':  0.04,
                'demand_cap_type':        'annual',
                'annual_demand_lbs':      600_000,
            },
            'Farmers Market': {
                'price_per_lb':           0.82,
                'packaging_cost_per_lb':  0.10,
                'demand_cap_type':        'weekly_varying',
                'demand_ranges': [
                    {'weeks': 'W25-W38', 'weekly_lbs': 3_000},
                ],
            },
        },
    },
    # ------------------------------------------------------------------
    {
        'name': 'Napa Cabbage',
        'days_to_maturity': 70,      # 10 weeks: 1 plant + 7 grow + 2 harvest
        'planting_weeks':   1,
        'harvest_weeks':    2,
        'seasons': [
            {'start': '03-01', 'end': '07-01'},   # spring
            {'start': '07-15', 'end': '11-01'},   # fall
        ],
        'min_acreage_if_planted':  2.0,
        'crop_diversity_max':      0.40,
        'yield_per_acre':          3_500,
        'crop_size_m3_per_lb':     0.0025,
        'planting_type':           'seed',
        'seed_transplant_cost_per_acre': 160,
        'labor_plant': 0.7,  'labor_grow': 0.5,  'labor_harv': 2.1,
        'water_plant': 10_400, 'water_grow': 15_000, 'water_harv': 12_700,
        'fert_plant':  0.46,  'fert_grow':  0.35,  'fert_harv':  0.23,
        'fuel_plant':  3.5,   'fuel_grow':  1.8,   'fuel_harv':  4.6,
        'channels': {
            'Wholesale': {
                'price_per_lb':           0.65,
                'packaging_cost_per_lb':  0.07,
                'demand_cap_type':        'annual',
                'annual_demand_lbs':      700_000,
            },
            'Farmers Market': {
                'price_per_lb':           1.25,
                'packaging_cost_per_lb':  0.14,
                'demand_cap_type':        'weekly_flat',
                'weekly_demand_lbs':      3_600,
            },
        },
    },
    # ------------------------------------------------------------------
    {
        'name': 'Hot Peppers',
        'days_to_maturity': 147,     # 21 weeks: 2 plant + 13 grow + 6 harvest
        'planting_weeks':   2,
        'harvest_weeks':    6,
        'seasons': [
            {'start': '04-01', 'end': '10-15'},
        ],
        'min_acreage_if_planted':  2.0,
        'crop_diversity_max':      0.55,
        'yield_per_acre':          834,
        'crop_size_m3_per_lb':     0.0018,
        'planting_type':           'transplant',
        'seed_transplant_cost_per_acre': 420,
        'labor_plant': 1.2,  'labor_grow': 0.7,  'labor_harv': 2.8,
        'water_plant': 16_200, 'water_grow': 23_100, 'water_harv': 19_600,
        'fert_plant':  0.69,  'fert_grow':  0.46,  'fert_harv':  0.35,
        'fuel_plant':  4.2,   'fuel_grow':  2.8,   'fuel_harv':  5.8,
        'channels': {
            'Wholesale': {
                'price_per_lb':           1.80,
                'packaging_cost_per_lb':  0.14,
                'demand_cap_type':        'annual',
                'annual_demand_lbs':      300_000,
            },
            'Farmers Market': {
                'price_per_lb':           3.20,
                'packaging_cost_per_lb':  0.25,
                'demand_cap_type':        'weekly_varying',
                'demand_ranges': [
                    {'weeks': 'W22-W44', 'weekly_lbs': 1_600},
                ],
            },
        },
    },
    # ------------------------------------------------------------------
    {
        'name': 'Garlic',
        'days_to_maturity': 196,     # 28 weeks: 3 plant + 22 grow + 3 harvest
        'planting_weeks':   3,
        'harvest_weeks':    3,
        'seasons': [
            {'start': '01-01', 'end': '08-01'},
        ],
        'min_acreage_if_planted':  5.0,
        'crop_diversity_max':      0.60,
        'yield_per_acre':          834,
        'crop_size_m3_per_lb':     0.0012,
        'planting_type':           'seed',
        'seed_transplant_cost_per_acre': 380,
        'labor_plant': 0.9,  'labor_grow': 0.5,  'labor_harv': 1.4,
        'water_plant': 13_900, 'water_grow': 9_200, 'water_harv': 8_100,
        'fert_plant':  0.46,  'fert_grow':  0.23,  'fert_harv':  0.12,
        'fuel_plant':  4.6,   'fuel_grow':  1.8,   'fuel_harv':  3.5,
        'channels': {
            'Wholesale': {
                'price_per_lb':           3.00,
                'packaging_cost_per_lb':  0.20,
                'demand_cap_type':        'annual',
                'annual_demand_lbs':      200_000,
            },
            'Farmers Market': {
                'price_per_lb':           5.20,
                'packaging_cost_per_lb':  0.40,
                'demand_cap_type':        'weekly_varying',
                'demand_ranges': [
                    {'weeks': 'W22-W35', 'weekly_lbs': 1_000},
                ],
            },
        },
    },
]

# Auto-generate and display succession count
_succs = generate_successions(CROPS)
print(f'Generated {len(_succs)} succession slots across {len(CROPS)} crops:')
for c_idx, crop in enumerate(CROPS):
    n = sum(1 for s in _succs if s['crop'] == c_idx)
    print(f'  {crop["name"]}: {n} slots')

### 3H. Variable Cost Unit Rates

In [ ]:
# ---- 3H. Variable Cost Unit Rates ----
COST_FERT_PER_LB  = 0.82   # $/lb fertilizer + pesticide blended
COST_FUEL_PER_L   = 1.25   # $/liter fuel (diesel)

### 3I. Monte Carlo Risk Parameters

Configure uncertainty for the Monte Carlo simulation (Section 8).

| Field | Meaning | Typical range |
|-------|---------|---------------|
| `yield_std_dev` | Yield uncertainty as a fraction of base yield | 0.10–0.25 |
| `price_std_dev` | Price uncertainty as a fraction of base price | 0.08–0.20 |
| `demand_scale_std` | Demand cap uncertainty as a fraction of base demand | 0.15–0.30 |

Set any value to `0.0` to hold that parameter fixed in simulation.
Set all to `0.0` (or remove the crop) to skip Monte Carlo entirely.

> **Assumption**: Uncertainties are sampled independently. Market-wide shocks
> (e.g. drought that simultaneously depresses yield *and* raises prices) are not
> modelled — each crop/channel draws from its own independent distribution.

In [ ]:
# ---- 3I. Monte Carlo Risk Parameters ----
# Edit the values below; do NOT change the crop names (must match Section 3G).
MC_RISK = {
    'Spinach': {
        'yield_std_dev': 0.15,       # ±15% yield variability
        'Wholesale':      {'price_std_dev': 0.10, 'demand_scale_std': 0.20},
        'Farmers Market': {'price_std_dev': 0.12, 'demand_scale_std': 0.25},
    },
    'Cucumbers': {
        'yield_std_dev': 0.18,
        'Wholesale':      {'price_std_dev': 0.10, 'demand_scale_std': 0.20},
        'Farmers Market': {'price_std_dev': 0.12, 'demand_scale_std': 0.25},
    },
    'Sweet Corn': {
        'yield_std_dev': 0.20,
        'Wholesale':      {'price_std_dev': 0.08, 'demand_scale_std': 0.15},
        'Farmers Market': {'price_std_dev': 0.10, 'demand_scale_std': 0.20},
    },
    'Napa Cabbage': {
        'yield_std_dev': 0.15,
        'Wholesale':      {'price_std_dev': 0.10, 'demand_scale_std': 0.20},
        'Farmers Market': {'price_std_dev': 0.12, 'demand_scale_std': 0.25},
    },
    'Hot Peppers': {
        'yield_std_dev': 0.22,       # higher variance — specialty crop
        'Wholesale':      {'price_std_dev': 0.15, 'demand_scale_std': 0.25},
        'Farmers Market': {'price_std_dev': 0.18, 'demand_scale_std': 0.30},
    },
    'Garlic': {
        'yield_std_dev': 0.12,       # relatively stable
        'Wholesale':      {'price_std_dev': 0.12, 'demand_scale_std': 0.20},
        'Farmers Market': {'price_std_dev': 0.15, 'demand_scale_std': 0.25},
    },
}

# Merge MC_RISK into CROPS (safe — does not overwrite existing crop keys)
for _crop in CROPS:
    _risk = MC_RISK.get(_crop['name'], {})
    _crop['yield_std_dev'] = _risk.get('yield_std_dev', 0.0)
    for _ch_name, _ch_risk in _risk.items():
        if _ch_name in _crop.get('channels', {}):
            _crop['channels'][_ch_name].update(_ch_risk)

_n_uncertain = sum(1 for c in CROPS if c.get('yield_std_dev', 0) > 0)
print(f'MC risk configured for {_n_uncertain}/{len(CROPS)} crops.')
print('  Crop                yield_std   channels configured')
for _c in CROPS:
    _ch_str = ', '.join(
        f"{ch}(p={_c['channels'][ch].get('price_std_dev',0):.0%} d={_c['channels'][ch].get('demand_scale_std',0):.0%})"
        for ch in _c.get('channels', {})
        if _c['channels'][ch].get('price_std_dev', 0) > 0 or _c['channels'][ch].get('demand_scale_std', 0) > 0
    )
    print(f'  {_c["name"]:<18} {_c.get("yield_std_dev", 0):.0%}       {_ch_str}')

---
## Section 4: Validate & Solve

Run the two cells below.  The first checks your inputs for errors or
suspicious values.  The second runs the MILP solver (typically < 5 seconds).

In [ ]:
# ---- Validate inputs ----
warnings_list = validate_params(
    total_land=TOTAL_LAND,
    harvest_storage_cap=HARVEST_STORAGE_CAP,
    opening_cash=OPENING_CASH,
    credit_line=CREDIT_LINE,
    operating_weeks=OPERATING_WEEKS,
    crops=CROPS,
    channels=CHANNELS,
    rainfall_weekly_L_per_acre=RAINFALL_WEEKLY,
    year_round_employees=YEAR_ROUND_EMPLOYEES,
    seasonal_employees=SEASONAL_EMPLOYEES,
    city_water_enabled=CITY_WATER_ENABLED,
    well_enabled=WELL_ENABLED,
    catchment_enabled=CATCHMENT_ENABLED,
    catchment_surface_area_m2=CATCHMENT_SURFACE_AREA_M2,
    well_weekly_cap_L=WELL_WEEKLY_CAP_L,
    max_plant_acres_per_week=MAX_PLANT_ACRES_PER_WEEK,
    max_harvest_acres_per_week=MAX_HARVEST_ACRES_PER_WEEK,
    year=2024,
)
if warnings_list:
    for w in warnings_list:
        print(f'WARNING: {w}')
else:
    print('Validation passed — no warnings.')

In [ ]:
# ---- Run solver ----
_SOLVER_KWARGS = dict(
    total_land=TOTAL_LAND,
    opening_cash=OPENING_CASH,
    credit_line=CREDIT_LINE,
    harvest_storage_cap=HARVEST_STORAGE_CAP,
    equip_storage=EQUIP_STORAGE,
    fuel_storage=FUEL_STORAGE,
    water_storage_capacity=WATER_STORAGE_CAPACITY,
    operating_weeks=OPERATING_WEEKS,
    rate_harvest_storage=RATE_HARVEST_STORAGE,
    rate_water_storage=RATE_WATER_STORAGE,
    rate_irrigation=RATE_IRRIGATION,
    rate_equip_storage=RATE_EQUIP_STORAGE,
    rate_fuel_storage=RATE_FUEL_STORAGE,
    rate_machinery=RATE_MACHINERY,
    rainfall_weekly_L_per_acre=RAINFALL_WEEKLY,
    catchment_enabled=CATCHMENT_ENABLED,
    catchment_surface_area_m2=CATCHMENT_SURFACE_AREA_M2,
    well_enabled=WELL_ENABLED,
    well_weekly_cap_L=WELL_WEEKLY_CAP_L,
    well_cost_per_L=WELL_COST_PER_L,
    city_water_enabled=CITY_WATER_ENABLED,
    city_water_cost_per_L=CITY_WATER_COST_PER_L,
    city_water_fixed_fee_per_week=CITY_WATER_FIXED_FEE_PER_WEEK,
    year_round_employees=YEAR_ROUND_EMPLOYEES,
    seasonal_employees=SEASONAL_EMPLOYEES,
    overtime_enabled=OVERTIME_ENABLED,
    overtime_threshold_hrs=OVERTIME_THRESHOLD_HRS,
    overtime_rate_multiplier=OVERTIME_RATE_MULTIPLIER,
    max_plant_acres_per_week=MAX_PLANT_ACRES_PER_WEEK,
    max_harvest_acres_per_week=MAX_HARVEST_ACRES_PER_WEEK,
    cost_fert_per_lb=COST_FERT_PER_LB,
    cost_fuel_per_L=COST_FUEL_PER_L,
    crops=CROPS,
    channels=CHANNELS,
    year=2024,
)

result = solve_farm_v2(**_SOLVER_KWARGS)

print(f'Solver status   : {result["status"]}')
if result['status'] == 'optimal':
    print(f'Annual profit   : ${result["profit"]:>12,.2f}')
    print(f'Annual revenue  : ${result["revenue"]:>12,.2f}')
    print(f'Variable cost   : ${result["variable_cost"]:>12,.2f}')
    print(f'Fixed cost      : ${result["fixed_cost"]:>12,.2f}')
    print(f'Peak borrowing  : ${result["peak_borrowing"]:>12,.2f}')

---
## Section 5: Results

The cells below display and chart the optimal solution.

In [ ]:
# ---- 5A. Optimal Planting Schedule ----
if result['status'] != 'optimal':
    print('No optimal solution found — cannot show schedule.')
else:
    succs    = result['successions']
    crops    = result['crops']
    x        = result['x']
    y        = result['y']
    rows = []
    for i, s in enumerate(succs):
        if x[i] > 0.005:
            crop = crops[s['crop']]
            acres   = x[i] * TOTAL_LAND
            start_d = week_start_date(s['start_week'])
            end_d   = week_end_date(s['start_week'] + len(s['weeks']) - 1)
            rows.append({
                'Crop':        crop['name'],
                'Succession':  s['name'],
                'Start date':  start_d.strftime('%b %d'),
                'End date':    end_d.strftime('%b %d'),
                'Weeks':       len(s['weeks']),
                'Acres':       round(acres, 1),
                'Land %':      f'{x[i]*100:.1f}%',
                'Planted':     'Yes' if y[i] > 0.5 else 'No',
            })
    df_plan = pd.DataFrame(rows).sort_values(['Crop', 'Start date'])
    display(df_plan.to_string(index=False))

In [ ]:
# ---- 5B. Channel Sales Summary ----
if result['status'] == 'optimal':
    v    = result['v']   # (n_crops, n_ch, 53)
    channels = result['channels']
    rows = []
    for c_idx, crop in enumerate(CROPS):
        for ch_idx, ch in enumerate(channels):
            sold = v[c_idx, ch_idx, 1:].sum()
            if sold > 0.5:
                ch_spec = crop['channels'].get(ch['name'], {})
                price   = ch_spec.get('price_per_lb', 0)
                pkg     = ch_spec.get('packaging_cost_per_lb', 0)
                rows.append({
                    'Crop':        crop['name'],
                    'Channel':     ch['name'],
                    'Lbs sold':    f'{sold:,.0f}',
                    '$/lb':        f'${price:.2f}',
                    'Pkg $/lb':    f'${pkg:.2f}',
                    'Net revenue': f'${sold*(price-pkg):,.0f}',
                })
    df_sales = pd.DataFrame(rows)
    display(df_sales.to_string(index=False))

---
## Section 6: Reporting Charts

### Chart 1 — Profit Waterfall

In [ ]:
if result['status'] == 'optimal':
    rev  = result['revenue']
    vcst = result['variable_cost']
    fcst = result['fixed_cost']
    prof = result['profit']

    # Decompose variable cost for the waterfall
    from farm_v2.utils.labor import compute_labor_arrays
    labor_res = compute_labor_arrays(YEAR_ROUND_EMPLOYEES, SEASONAL_EMPLOYEES)

    # Approximate breakdown: labor / input (fert+fuel+water+seed) / packaging
    x_sol    = result['x']
    succs_list = result['successions']
    from farm_v2.utils.demand import build_demand_table
    from farm_v2.utils.water import compute_net_water_rates
    from farm_v2.utils.labor import compute_labor_rates

    net_wr   = compute_net_water_rates(succs_list, CROPS, RAINFALL_WEEKLY)
    lab_rate = compute_labor_rates(succs_list, CROPS)
    hr_rate  = labor_res['hourly_labor_rate']

    labor_vcost = 0.0
    input_vcost = 0.0
    for i, s in enumerate(succs_list):
        crop = CROPS[s['crop']]
        for w, ph in zip(s['weeks'], s['phases']):
            if not (1 <= w <= 52): continue
            labor_vcost += x_sol[i] * TOTAL_LAND * lab_rate[i][w] * hr_rate[w-1]
            input_vcost += x_sol[i] * TOTAL_LAND * net_wr[i][w] * WELL_COST_PER_L
            fk = ['fert_plant','fert_grow','fert_harv'][ph]
            uk = ['fuel_plant','fuel_grow','fuel_harv'][ph]
            input_vcost += x_sol[i] * TOTAL_LAND * crop[fk] * COST_FERT_PER_LB
            input_vcost += x_sol[i] * TOTAL_LAND * crop[uk] * COST_FUEL_PER_L
        input_vcost += x_sol[i] * TOTAL_LAND * crop.get('seed_transplant_cost_per_acre', 0)

    pkg_cost  = vcst - labor_vcost - input_vcost   # packaging residual
    sal_cost  = labor_res['salaried_weekly_cost'] * 52
    infra_cost = fcst - sal_cost

    labels  = ['Revenue', 'Packaging', 'Variable Labor', 'Inputs', 'Infra & Equipment', 'Salaried Labor', 'Net Profit']
    values  = [rev, -pkg_cost, -labor_vcost, -input_vcost, -infra_cost, -sal_cost, prof]
    colors  = ['#2196F3', '#FF7043', '#FF9800', '#FFC107', '#78909C', '#90A4AE',
               '#4CAF50' if prof >= 0 else '#F44336']

    running = 0
    bottoms = []
    for v_val in values:
        bottoms.append(running)
        running += v_val

    fig, ax = plt.subplots(figsize=(11, 5))
    for idx, (lbl, val, bot, col) in enumerate(zip(labels, values, bottoms, colors)):
        bar_bot = min(bot, bot + val)
        bar_h   = abs(val)
        ax.bar(idx, bar_h, bottom=bar_bot, color=col, edgecolor='white', width=0.6)
        va   = 'bottom' if val >= 0 else 'top'
        ypos = bot + val + (50_000 if val >= 0 else -50_000)
        ax.text(idx, ypos, f'${abs(val)/1e6:.2f}M', ha='center', va=va, fontsize=8.5)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
    ax.set_title('Profit Waterfall — Annual', fontsize=13, fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('USD')
    plt.tight_layout()
    plt.show()

### Chart 2 — Weekly Land Allocation Heatmap

In [ ]:
if result['status'] == 'optimal':
    succs_list = result['successions']
    x_sol      = result['x']
    crop_names = [c['name'] for c in CROPS]
    n_crops    = len(CROPS)

    # Build weekly acres per crop
    land_by_crop_week = np.zeros((n_crops, 52))
    for i, s in enumerate(succs_list):
        if x_sol[i] < 0.001: continue
        acres = x_sol[i] * TOTAL_LAND
        for w in s['weeks']:
            if 1 <= w <= 52:
                land_by_crop_week[s['crop'], w-1] += acres

    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(land_by_crop_week, aspect='auto', cmap='YlGn',
                   vmin=0, vmax=TOTAL_LAND * 0.6, interpolation='nearest')
    ax.set_yticks(range(n_crops))
    ax.set_yticklabels(crop_names)
    ax.set_xlabel('Week of Year')
    ax.set_title('Land Allocation by Crop and Week (acres)', fontsize=13, fontweight='bold')
    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    month_starts = [0, 4, 8, 13, 17, 21, 26, 30, 35, 39, 43, 47]
    ax.set_xticks(month_starts)
    ax.set_xticklabels(months)
    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
    cbar.set_label('Acres')
    plt.tight_layout()
    plt.show()

### Chart 3 — Weekly Revenue by Crop and Channel

In [ ]:
if result['status'] == 'optimal':
    v_sol    = result['v']    # (n_crops, n_ch, 53)
    channels = result['channels']
    crop_names = [c['name'] for c in CROPS]

    # Net revenue (price - packaging) per crop per week
    weekly_rev = np.zeros((len(CROPS), 52))
    for c_idx, crop in enumerate(CROPS):
        for ch_idx, ch in enumerate(channels):
            ch_spec = crop['channels'].get(ch['name'], {})
            net     = ch_spec.get('price_per_lb', 0) - ch_spec.get('packaging_cost_per_lb', 0)
            for w in range(1, 53):
                weekly_rev[c_idx, w-1] += v_sol[c_idx, ch_idx, w] * net

    palette = ['#1565C0','#2E7D32','#E65100','#6A1B9A','#AD1457','#00695C']
    fig, ax = plt.subplots(figsize=(14, 5))
    bottom = np.zeros(52)
    weeks  = np.arange(1, 53)
    for c_idx, (name, col) in enumerate(zip(crop_names, palette)):
        ax.bar(weeks, weekly_rev[c_idx], bottom=bottom, label=name, color=col,
               width=0.9, linewidth=0)
        bottom += weekly_rev[c_idx]

    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    month_starts = [1, 5, 9, 14, 18, 22, 27, 31, 36, 40, 44, 48]
    ax.set_xticks(month_starts)
    ax.set_xticklabels(months)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.set_ylabel('Net Revenue ($ thousand)')
    ax.set_title('Weekly Net Revenue by Crop', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', ncol=3, fontsize=8)
    plt.tight_layout()
    plt.show()

### Chart 4 — Weekly Water Supply vs. Demand

In [ ]:
if result['status'] == 'optimal':
    from farm_v2.utils.water import (compute_free_water, compute_net_water_rates,
                                      compute_catchment_supply, compute_well_supply)

    succs_list = result['successions']
    x_sol      = result['x']
    nwr        = compute_net_water_rates(succs_list, CROPS, RAINFALL_WEEKLY)

    # Weekly water demand
    water_demand = np.zeros(52)
    for i, s in enumerate(succs_list):
        if x_sol[i] < 1e-6: continue
        for w in s['weeks']:
            if 1 <= w <= 52:
                water_demand[w-1] += x_sol[i] * TOTAL_LAND * nwr[i][w]

    # Supply components
    catchment = compute_catchment_supply(RAINFALL_WEEKLY, CATCHMENT_SURFACE_AREA_M2)                 if CATCHMENT_ENABLED else np.zeros(52)
    well      = compute_well_supply(WELL_ENABLED, WELL_WEEKLY_CAP_L)
    rainfall_field = RAINFALL_WEEKLY * TOTAL_LAND  # total rainfall on planted field (informational)

    weeks = np.arange(1, 53)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(weeks, 0, well/1e6, alpha=0.6, color='#1565C0', label='Well supply')
    ax.fill_between(weeks, well/1e6, (well+catchment)/1e6, alpha=0.5,
                    color='#42A5F5', label='Catchment supply')
    ax.plot(weeks, water_demand/1e6, color='#D32F2F', linewidth=2,
            label='Crop water demand', zorder=5)
    ax.plot(weeks, (well+catchment)/1e6, color='#1565C0', linewidth=1.2,
            linestyle='--', label='Total free supply')

    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    month_starts = [1, 5, 9, 14, 18, 22, 27, 31, 36, 40, 44, 48]
    ax.set_xticks(month_starts)
    ax.set_xticklabels(months)
    ax.set_ylabel('Million Liters / Week')
    ax.set_title('Water Supply vs. Crop Demand by Week', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

### Chart 5 — Weekly Labor Utilization

In [ ]:
if result['status'] == 'optimal':
    from farm_v2.utils.labor import compute_labor_arrays, compute_labor_rates

    succs_list = result['successions']
    x_sol      = result['x']
    lab_result = compute_labor_arrays(YEAR_ROUND_EMPLOYEES, SEASONAL_EMPLOYEES)
    lab_rate   = compute_labor_rates(succs_list, CROPS)

    capacity_total  = lab_result['labor_capacity']          # (52,)
    capacity_hourly = lab_result['labor_capacity_hourly']   # (52,)

    # Compute labor used per week
    labor_used = np.zeros(52)
    for i, s in enumerate(succs_list):
        if x_sol[i] < 1e-6: continue
        for w in s['weeks']:
            if 1 <= w <= 52:
                labor_used[w-1] += x_sol[i] * TOTAL_LAND * lab_rate[i][w]

    # Salaried contribution (constant)
    sal_cap = capacity_total - capacity_hourly   # salaried hours

    weeks = np.arange(1, 53)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(weeks, 0, sal_cap, alpha=0.3, color='#78909C', label='Salaried capacity')
    ax.fill_between(weeks, sal_cap, capacity_total, alpha=0.4,
                    color='#42A5F5', label='Hourly capacity')
    ax.plot(weeks, labor_used, color='#E65100', linewidth=2, label='Labor used', zorder=5)

    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    month_starts = [1, 5, 9, 14, 18, 22, 27, 31, 36, 40, 44, 48]
    ax.set_xticks(month_starts)
    ax.set_xticklabels(months)
    ax.set_ylabel('Labor Hours / Week')
    ax.set_title('Labor Capacity vs. Utilization by Week', fontsize=13, fontweight='bold')
    ax.legend()
    # Highlight over-capacity weeks
    over = labor_used > capacity_total
    if over.any():
        ax.fill_between(weeks, capacity_total, labor_used, where=over,
                        color='#D32F2F', alpha=0.4, label='Over capacity')
    plt.tight_layout()
    plt.show()

### Chart 6 — Weekly Cash Flow & Working Capital

In [ ]:
if result['status'] == 'optimal':
    wcf   = result['weekly_cash_flow'][1:]    # weeks 1-52
    peak  = result['peak_borrowing']
    weeks = np.arange(1, 53)

    fig, ax = plt.subplots(figsize=(14, 5))
    # Shade borrowing region (below zero)
    ax.fill_between(weeks, np.minimum(wcf, 0), 0,
                    color='#EF9A9A', alpha=0.6, label='Credit drawn')
    ax.fill_between(weeks, 0, np.maximum(wcf, 0),
                    color='#A5D6A7', alpha=0.5, label='Positive cash')
    ax.plot(weeks, wcf, color='#1B5E20', linewidth=2, label='Cumulative cash')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(-CREDIT_LINE, color='#B71C1C', linewidth=1.2,
               linestyle=':', label=f'Credit limit (-${CREDIT_LINE:,.0f})')

    if peak > 0:
        trough_w = int(np.argmin(wcf)) + 1
        ax.annotate(f'Peak draw: ${peak:,.0f}',
                    xy=(trough_w, -peak),
                    xytext=(trough_w + 3, -peak * 0.7),
                    arrowprops=dict(arrowstyle='->', color='#B71C1C'),
                    color='#B71C1C', fontsize=9)
        weeks_neg = int((wcf < 0).sum())
        est_interest = peak * ANNUAL_INTEREST_RATE * (weeks_neg / 52)
        print(f'Peak credit draw : ${peak:,.0f} (week {trough_w})')
        print(f'Weeks underwater : {weeks_neg}')
        print(f'Est. interest    : ${est_interest:,.0f}  '
              f'(peak × {ANNUAL_INTEREST_RATE*100:.0f}% × {weeks_neg}/52 wks)')
    else:
        print('Farm remains cash-positive all year — no credit required.')

    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    month_starts = [1, 5, 9, 14, 18, 22, 27, 31, 36, 40, 44, 48]
    ax.set_xticks(month_starts)
    ax.set_xticklabels(months)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.set_ylabel('Cumulative Cash Position')
    ax.set_title('Weekly Cash Flow — Operating Credit Requirement', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Section 7: Sensitivity Analysis

Runs the optimizer repeatedly over a range of values for key decision
parameters.  Each sweep takes 20–60 seconds.  Set a flag to `False` to skip.

**Model assumptions note:** This model optimizes a single farm's planting
schedule assuming fixed per-crop yields. It does not model market-wide supply
shocks (e.g., drought affecting all farms) that would simultaneously reduce
yields and raise market prices. Yield and price are treated as independent.
See the Monte Carlo section (Section 8) for yield and price uncertainty.

In [ ]:
# ---- Sensitivity toggles ----
RUN_LABOR_SWEEP  = True   # seasonal crew headcount vs. profit
RUN_CREDIT_SWEEP = True   # credit line availability vs. feasibility
RUN_BINDING_SWEEP = True  # auto-detect and sweep the tightest resource constraint

In [ ]:
import copy, time

def _run_sweep(param_name, values, kwarg_override_fn, label_fn=None):
    '''Run solve_farm_v2 for each value, return (values, profits, statuses).'''
    profits, statuses = [], []
    base_kwargs = dict(_SOLVER_KWARGS)
    for val in values:
        kw = kwarg_override_fn(copy.deepcopy(base_kwargs), val)
        r  = solve_farm_v2(**kw)
        profits.append(r['profit'] if r['status'] == 'optimal' else None)
        statuses.append(r['status'])
        print(f'  {param_name}={val} → {r["status"]}'
              + (f', profit=${r["profit"]:,.0f}' if r['status']=='optimal' else ''))
    return values, profits, statuses

def _plot_sweep(ax, xs, ys, xlabel, title, base_val=None):
    valid = [(x, y) for x, y in zip(xs, ys) if y is not None]
    if not valid: ax.text(0.5, 0.5, 'No optimal solutions', ha='center', transform=ax.transAxes); return
    xv, yv = zip(*valid)
    ax.plot(xv, yv, 'o-', color='#1565C0', linewidth=2, markersize=6)
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    if base_val is not None and base_val in xv:
        idx = xv.index(base_val)
        ax.axvline(base_val, color='orange', linewidth=1.2, linestyle=':', label='Base case')
        ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.set_xlabel(xlabel); ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Annual Profit')

In [ ]:
# ---- Sweep 1: Seasonal labor crew size ----
if RUN_LABOR_SWEEP and result['status'] == 'optimal':
    print('Running labor headcount sweep...')
    # Vary the spring/summer crew size from 10 to 40
    crew_sizes = list(range(10, 45, 5))
    base_count = SEASONAL_EMPLOYEES[0]['count']

    def _set_crew(kw, n):
        kw['seasonal_employees'] = [
            dict(e, count=n) if 'spring' in e['position'].lower() else e
            for e in kw['seasonal_employees']
        ]
        return kw

    t0 = time.time()
    cs, profits_labor, _ = _run_sweep('crew_size', crew_sizes, _set_crew)
    print(f'Done in {time.time()-t0:.1f}s')

    fig, ax = plt.subplots(figsize=(9, 4))
    _plot_sweep(ax, cs, profits_labor,
                'Spring/summer crew headcount', 'Profit vs. Seasonal Crew Size',
                base_val=base_count)
    plt.tight_layout(); plt.show()

In [ ]:
# ---- Sweep 2: Credit line availability ----
if RUN_CREDIT_SWEEP and result['status'] == 'optimal':
    print('Running credit line sweep...')
    credit_vals = [0, 100_000, 200_000, 300_000, 400_000, 500_000, 700_000, 1_000_000]

    def _set_credit(kw, cl):
        kw['credit_line'] = cl
        return kw

    t0 = time.time()
    cv, profits_credit, statuses_credit = _run_sweep('credit_line', credit_vals, _set_credit)
    print(f'Done in {time.time()-t0:.1f}s')

    fig, ax = plt.subplots(figsize=(9, 4))
    _plot_sweep(ax, [c/1e3 for c in cv], profits_credit,
                'Credit line ($k)', 'Profit vs. Credit Line',
                base_val=CREDIT_LINE/1e3)
    infeasible = [c/1e3 for c, s in zip(cv, statuses_credit) if s != 'optimal']
    if infeasible:
        for x in infeasible:
            ax.axvline(x, color='red', alpha=0.3, linewidth=4)
        print(f'Infeasible at credit lines: {[f"${c:.0f}k" for c in infeasible]}')
    plt.tight_layout(); plt.show()

In [ ]:
# ---- Sweep 3: Auto-detect binding constraint ----
if RUN_BINDING_SWEEP and result['status'] == 'optimal':
    # Identify which resource has the least slack in the base solution
    # by checking constraint slack for water, labor, harvest storage
    from farm_v2.utils.water import compute_free_water, compute_net_water_rates
    from farm_v2.utils.labor import compute_labor_arrays, compute_labor_rates

    succs = result['successions']
    x_sol = result['x']
    lab_res = compute_labor_arrays(YEAR_ROUND_EMPLOYEES, SEASONAL_EMPLOYEES)
    lab_rate = compute_labor_rates(succs, CROPS)
    nwr = compute_net_water_rates(succs, CROPS, RAINFALL_WEEKLY)
    free_w = compute_free_water(RAINFALL_WEEKLY, CATCHMENT_ENABLED,
                                CATCHMENT_SURFACE_AREA_M2, 0.85,
                                WELL_ENABLED, WELL_WEEKLY_CAP_L)

    water_slack  = np.zeros(52)
    labor_slack  = np.zeros(52)
    for w in range(1, 53):
        wd = sum(x_sol[i]*TOTAL_LAND*nwr[i][w] for i in range(len(succs)))
        ld = sum(x_sol[i]*TOTAL_LAND*lab_rate[i][w] for i in range(len(succs)))
        water_slack[w-1]  = free_w[w-1] - wd
        labor_slack[w-1]  = lab_res['labor_capacity'][w-1] - ld

    water_min_slack = water_slack.min() / (free_w.mean() + 1e-9)
    labor_min_slack = labor_slack.min() / (lab_res['labor_capacity'].mean() + 1e-9)

    if water_min_slack < labor_min_slack:
        binding = 'water'
        pct_slack = water_min_slack * 100
        print(f'Binding constraint: WATER (tightest slack = {pct_slack:.1f}% of avg supply)')
        sweep_vals = [v * WELL_WEEKLY_CAP_L for v in [0.5, 0.7, 0.85, 1.0, 1.25, 1.5, 2.0]]
        def _set_binding(kw, v): kw['well_weekly_cap_L'] = v; return kw
        xlabel = 'Well capacity (L/week)'
        base_bv = WELL_WEEKLY_CAP_L
    else:
        binding = 'labor'
        pct_slack = labor_min_slack * 100
        print(f'Binding constraint: LABOR (tightest slack = {pct_slack:.1f}% of avg capacity)')
        sweep_vals = [int(SEASONAL_EMPLOYEES[0]['count'] * m)
                      for m in [0.5, 0.7, 0.85, 1.0, 1.25, 1.5, 2.0]]
        def _set_binding(kw, v):
            kw['seasonal_employees'] = [
                dict(e, count=v) if 'spring' in e['position'].lower() else e
                for e in kw['seasonal_employees']
            ]
            return kw
        xlabel = 'Spring crew headcount'
        base_bv = SEASONAL_EMPLOYEES[0]['count']

    t0 = time.time()
    bv, profits_bind, _ = _run_sweep(binding, sweep_vals, _set_binding)
    print(f'Done in {time.time()-t0:.1f}s')

    fig, ax = plt.subplots(figsize=(9, 4))
    _plot_sweep(ax, bv, profits_bind, xlabel,
                f'Profit vs. Binding Constraint ({binding.title()})',
                base_val=base_bv)
    plt.tight_layout(); plt.show()

---
## Section 8: Monte Carlo Risk Analysis

Simulates farm profit under uncertain yields, prices, and demand.

**How it works (Scenario A — fixed plan):**
1. The optimal planting plan (`x[i]`) from Section 4 is held fixed — the
   farmer has already committed to planting.
2. For each simulation run, yield, price, and demand are independently
   sampled from Normal distributions using the `_std_dev` fields you set
   in the crop definitions (Section 3G).
3. Actual sales = min(what was grown, sampled demand cap).
4. Profit is recalculated.  Costs are held fixed (the farm already spent them).

**Assumptions note:** Yields and prices are sampled independently.  In
commodity markets they are negatively correlated (bad harvest → higher prices),
but for a small local/regional farm this effect is weak.  Market-wide supply
shocks are not modelled.

In [ ]:
# ---- Monte Carlo toggles ----
MC_ENABLED  = True
MC_N_RUNS   = 300     # increase to 500–1000 for smoother distribution (slower)
MC_SEED     = 42      # set to None for non-deterministic runs

In [ ]:
import copy as _copy

def _run_monte_carlo(result, crops, channels, n_runs, seed):
    '''
    Fixed-plan MC: hold x[i] and v[c,ch,w] from the base solve,
    resample yield/price/demand, return array of profit outcomes.
    '''
    rng        = np.random.default_rng(seed)
    x_sol      = result['x']
    v_base     = result['v']         # (n_crops, n_ch, 53)
    succs      = result['successions']
    base_rev   = result['revenue']
    base_vcost = result['variable_cost']
    base_fcost = result['fixed_cost']

    n_crops    = len(crops)
    n_channels = len(channels)
    profits    = []

    for _ in range(n_runs):
        # Sample yield multipliers (one per crop)
        yield_mult = np.ones(n_crops)
        for c_idx, crop in enumerate(crops):
            base_y = crop['yield_per_acre']
            std_y  = crop.get('yield_std_dev', 0.0)
            if std_y > 0:
                sampled = rng.normal(base_y, std_y)
                yield_mult[c_idx] = max(0.0, sampled) / base_y

        # Sample price multipliers and demand scale per crop-channel
        price_mult  = np.ones((n_crops, n_channels))
        demand_mult = np.ones((n_crops, n_channels))
        for c_idx, crop in enumerate(crops):
            for ch_idx, ch in enumerate(channels):
                ch_spec = crop.get('channels', {}).get(ch['name'], {})
                base_p  = ch_spec.get('price_per_lb', 0.0)
                std_p   = ch_spec.get('price_std_dev', 0.0)
                if std_p > 0 and base_p > 0:
                    sampled_p = rng.normal(base_p, std_p)
                    price_mult[c_idx, ch_idx] = max(0.0, sampled_p) / base_p
                d_std = ch_spec.get('demand_scale_std', 0.0)
                if d_std > 0:
                    demand_mult[c_idx, ch_idx] = max(0.0, rng.normal(1.0, d_std))

        # Compute actual revenue under sampled conditions
        revenue = 0.0
        for c_idx, crop in enumerate(crops):
            for ch_idx, ch in enumerate(channels):
                ch_spec  = crop.get('channels', {}).get(ch['name'], {})
                base_p   = ch_spec.get('price_per_lb', 0.0)
                pkg      = ch_spec.get('packaging_cost_per_lb', 0.0)
                s_price  = base_p * price_mult[c_idx, ch_idx]
                s_demand = demand_mult[c_idx, ch_idx]
                for w in range(1, 53):
                    # Actual harvest scales with yield mult
                    v_grown   = v_base[c_idx, ch_idx, w] * yield_mult[c_idx]
                    # Actual sales capped by sampled demand (demand_mult scales the base cap)
                    v_sold    = v_grown * s_demand
                    revenue  += v_sold * (s_price - pkg)

        profits.append(revenue - base_vcost - base_fcost)

    return np.array(profits)


if MC_ENABLED and result['status'] == 'optimal':
    # Check whether any stochastic fields are populated
    _has_stoch = any(
        c.get('yield_std_dev', 0) > 0 or
        any(c.get('channels', {}).get(ch['name'], {}).get('price_std_dev', 0) > 0 or
            c.get('channels', {}).get(ch['name'], {}).get('demand_scale_std', 0) > 0
            for ch in CHANNELS)
        for c in CROPS
    )
    if not _has_stoch:
        print('MC skipped — no yield_std_dev / price_std_dev / demand_scale_std '
              'fields set in CROPS.  Add them to Section 3G to enable risk analysis.')
    else:
        import time as _t
        t0 = _t.time()
        mc_profits = _run_monte_carlo(result, CROPS, CHANNELS, MC_N_RUNS, MC_SEED)
        print(f'MC complete: {MC_N_RUNS} runs in {_t.time()-t0:.1f}s')
        p10, p50, p90 = np.percentile(mc_profits, [10, 50, 90])
        base_profit   = result['profit']
        print(f'Base (deterministic) profit : ${base_profit:>10,.0f}')
        print(f'P10 (bad year)              : ${p10:>10,.0f}')
        print(f'P50 (median)                : ${p50:>10,.0f}')
        print(f'P90 (good year)             : ${p90:>10,.0f}')
        prob_loss = (mc_profits < 0).mean() * 100
        print(f'Probability of loss         : {prob_loss:.1f}%')

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.hist(mc_profits/1e3, bins=40, color='#1565C0', alpha=0.7, edgecolor='white')
        for val, lbl, col in [(p10,'P10','#EF5350'), (p50,'P50','#FB8C00'),
                               (p90,'P90','#43A047'), (base_profit,'Base','black')]:
            ax.axvline(val/1e3, color=col, linewidth=2,
                       linestyle='--' if lbl == 'Base' else '-', label=f'{lbl}: ${val/1e3:.0f}k')
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel('Annual Profit ($k)')
        ax.set_ylabel('Simulation count')
        ax.set_title(f'Monte Carlo Profit Distribution ({MC_N_RUNS} runs)', fontsize=13, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    if not MC_ENABLED:
        print('MC disabled (MC_ENABLED = False).')